# 🧪 Model Experimentation — BTC Next-Hour Return

**Goal:** Try several models, track every run on **DagsHub via MLflow**, and pick the
best one to fine-tune later.

**What we predict:** `target` = next-hour **return** (% change), not raw price.
So this is a **regression** problem — we score with **RMSE / MAE / R²**, not accuracy.

**Data:** the four split files from `data/processed/` (chronological split, no leakage):
- `X_train` (13,995 × 14), `y_train`
- `X_test`  (3,499 × 14),  `y_test`

## Plan
0. **Setup** — connect to DagsHub MLflow.
1. **Load** the split data.
2. **Baseline** — naive "no change" (`return = 0`). Every model must beat this.
3. **Models** — Linear, RandomForest, XGBoost… each logged to MLflow.
4. **Compare** — lowest RMSE wins → that one gets fine-tuned next.

> Auth: this notebook uses `dagshub.init()` (browser login). The final training
> script in `src/models/` will use a `.env` token instead (for automation).


In [3]:
import sys
from pathlib import Path

# Add project root so we can import our own `src` package
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

import mlflow
import dagshub

from src.config import CONFIG, get_path

# --- Connect to DagsHub's MLflow (opens a browser the first time to authorize) ---
dagshub.init(
    repo_owner="ar3080331",
    repo_name="Crypto-Price-Forecasting-MLops",
    mlflow=True,
)

# Name this experiment so all our runs group together on DagsHub
mlflow.set_experiment("btc-return-forecasting")

print("Connected. Tracking to:", mlflow.get_tracking_uri())


c:\Users\Ali Raza\Desktop\Crypto-Price-Forecasting-MLops\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

c:\Users\Ali Raza\Desktop\Crypto-Price-Forecasting-MLops\venv\Lib\site-packages\rich\live.py:260: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=0d55e740-8be2-48fe-91c2-82f6a9a8f9be&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=8d212a541855491e37179a3038ca51a89e25e60705d318fcc5029018df9171f2




Accessing as ar3080331

Initialized MLflow to track repo "ar3080331/Crypto-Price-Forecasting-MLops"

Repository ar3080331/Crypto-Price-Forecasting-MLops initialized!

2026/08/11 17:10:15 INFO mlflow.tracking.fluent: Experiment with name 'btc-return-forecasting' does not exist. Creating a new experiment.


Connected. Tracking to: https://dagshub.com/ar3080331/Crypto-Price-Forecasting-MLops.mlflow


## Step 1 — Load the split data

Read the four chronological split files. `X` = 14 features, `y` = the target return.


In [4]:
proc = get_path(CONFIG["data"]["processed_dir"])
sp = CONFIG["split"]

X_train = pd.read_parquet(proc / sp["x_train_file"])
y_train = pd.read_parquet(proc / sp["y_train_file"])["target"]
X_test  = pd.read_parquet(proc / sp["x_test_file"])
y_test  = pd.read_parquet(proc / sp["y_test_file"])["target"]

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_test :", X_test.shape,  "| y_test :", y_test.shape)
print("\nFeatures:", list(X_train.columns))


X_train: (13995, 14) | y_train: (13995,)
X_test : (3499, 14) | y_test : (3499,)

Features: ['return_1h', 'return_lag_1', 'return_lag_2', 'return_lag_3', 'return_lag_6', 'return_lag_12', 'return_lag_24', 'close_over_ma_6', 'volatility_6', 'close_over_ma_12', 'volatility_12', 'close_over_ma_24', 'volatility_24', 'log_volume']


## Step 2 — Baseline: naive "no change"

The simplest possible forecast: **predict a return of 0** for every hour
(i.e. "next price = current price"). It uses no features at all.

This is the **bar to beat**. Any real model that cannot beat this naive baseline
is worthless — it means our features add nothing. We log it to MLflow like any
other run, so DagsHub shows the comparison.

We score every model with the same function:
- **RMSE** — typical error (punishes big misses); lower is better.
- **MAE**  — average error size; lower is better.
- **R²**   — how much variance explained; higher is better (can be negative!).
- **Directional accuracy** — did we get up/down right? (bonus, higher is better).


In [5]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


def evaluate(y_true, y_pred):
    """Return the regression metrics we care about, as a dict."""
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
        "directional_acc": float((np.sign(y_true) == np.sign(y_pred)).mean()),
    }


# --- Baseline: predict 0 return for every test hour ---
baseline_pred = np.zeros(len(y_test))

with mlflow.start_run(run_name="baseline-naive"):
    mlflow.log_param("model", "naive_no_change")
    metrics = evaluate(y_test.values, baseline_pred)
    mlflow.log_metrics(metrics)

print("Baseline (naive no-change):")
for k, v in metrics.items():
    print(f"  {k:16}: {v:.6f}")


🏃 View run baseline-naive at: https://dagshub.com/ar3080331/Crypto-Price-Forecasting-MLops.mlflow/#/experiments/0/runs/5b71d6e94eab4ac3a02b6f65cc2a91d7
🧪 View experiment at: https://dagshub.com/ar3080331/Crypto-Price-Forecasting-MLops.mlflow/#/experiments/0
Baseline (naive no-change):
  rmse            : 0.004169
  mae             : 0.002746
  r2              : -0.000062
  directional_acc : 0.000000


## Step 3 — Try real models

We now train real models and log each to MLflow. Every model uses the **same 14
features** and the **same metrics**, so DagsHub can compare them fairly.

A small helper `run_experiment()` does the repetitive part for each model:
train → predict on test → compute metrics → log everything to MLflow.

Models we try:
- **LinearRegression** — simplest real model; a straight-line fit.
- **RandomForest** — many trees averaged; captures non-linear patterns.
- **XGBoost** — gradient-boosted trees; usually strongest on tabular data.

Reminder: each must beat the **naive baseline** RMSE to be worth anything.


In [7]:
pip install xgboost

  Using cached xgboost-3.4.0-py3-none-win_amd64.whl.metadata (2.0 kB)
Using cached xgboost-3.4.0-py3-none-win_amd64.whl (48.9 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

results = {}   # collect metrics so we can compare at the end


def run_experiment(name, model, params=None):
    """Train `model`, evaluate on the test set, and log the run to MLflow."""
    with mlflow.start_run(run_name=name):
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        metrics = evaluate(y_test.values, preds)

        mlflow.log_param("model", name)
        if params:
            mlflow.log_params(params)
        mlflow.log_metrics(metrics)

        results[name] = metrics
        print(f"{name:16} | RMSE {metrics['rmse']:.6f} | MAE {metrics['mae']:.6f} "
              f"| R2 {metrics['r2']:.4f} | Dir {metrics['directional_acc']:.3f}")
    return model


In [9]:
run_experiment("linear", LinearRegression())

run_experiment(
    "random_forest",
    RandomForestRegressor(n_estimators=200, max_depth=6, n_jobs=-1, random_state=42),
    params={"n_estimators": 200, "max_depth": 6},
)

run_experiment(
    "xgboost",
    XGBRegressor(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1,
    ),
    params={"n_estimators": 300, "max_depth": 4, "learning_rate": 0.05},
)


linear           | RMSE 0.004170 | MAE 0.002748 | R2 -0.0007 | Dir 0.497
🏃 View run linear at: https://dagshub.com/ar3080331/Crypto-Price-Forecasting-MLops.mlflow/#/experiments/0/runs/304162d2d2d54528b95e2f8dd4b27c11
🧪 View experiment at: https://dagshub.com/ar3080331/Crypto-Price-Forecasting-MLops.mlflow/#/experiments/0
random_forest    | RMSE 0.004165 | MAE 0.002748 | R2 0.0018 | Dir 0.508
🏃 View run random_forest at: https://dagshub.com/ar3080331/Crypto-Price-Forecasting-MLops.mlflow/#/experiments/0/runs/821e2d7645e6438d89bdfb95bd8c3bc2
🧪 View experiment at: https://dagshub.com/ar3080331/Crypto-Price-Forecasting-MLops.mlflow/#/experiments/0
xgboost          | RMSE 0.004186 | MAE 0.002753 | R2 -0.0083 | Dir 0.502
🏃 View run xgboost at: https://dagshub.com/ar3080331/Crypto-Price-Forecasting-MLops.mlflow/#/experiments/0/runs/ad6b57e63822469ab6d7e82f28102ca9
🧪 View experiment at: https://dagshub.com/ar3080331/Crypto-Price-Forecasting-MLops.mlflow/#/experiments/0


,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


## Step 4 — Fine-tune RandomForest (Optuna + TimeSeriesSplit)

RandomForest was our best model, so we tune its hyperparameters with **Optuna**.

Two things make this correct for a time series:
- **`TimeSeriesSplit`** for cross-validation — always trains on older folds and
  validates on newer ones (never shuffles), so no future leaks in.
- We score each trial by **RMSE** (lower is better) and let Optuna search.

We log the best result to MLflow. *Realistic expectation:* the data is close to a
random walk, so tuning will only nudge the score slightly — the point is a
correct, tracked tuning process, not a miracle model.


In [11]:
pip install optuna

  Using cached optuna-4.9.0-py3-none-any.whl.metadata (15 kB)
  Using cached colorlog-6.12.0-py3-none-any.whl.metadata (11 kB)
Using cached optuna-4.9.0-py3-none-any.whl (425 kB)
Using cached colorlog-6.12.0-py3-none-any.whl (12 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import optuna
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)  # keep output clean

tscv = TimeSeriesSplit(n_splits=4)   # time-aware CV


def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 400, step=50),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 50),
        "max_features": trial.suggest_float("max_features", 0.3, 1.0),
    }
    model = RandomForestRegressor(**params, n_jobs=-1, random_state=42)

    # cross_val_score returns negative RMSE (sklearn maximises) -> flip sign
    neg_rmse = cross_val_score(
        model, X_train, y_train,
        cv=tscv, scoring="neg_root_mean_squared_error", n_jobs=-1,
    ).mean()
    return -neg_rmse   # Optuna minimises RMSE


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=25, show_progress_bar=True)

print("Best CV RMSE:", round(study.best_value, 6))
print("Best params :", study.best_params)


Best trial: 15. Best value: 0.00489967: 100%|██████████| 25/25 [05:32<00:00, 13.30s/it]

Best CV RMSE: 0.0049
Best params : {'n_estimators': 250, 'max_depth': 3, 'min_samples_leaf': 34, 'max_features': 0.5649947641875525}


In [13]:
# Train the tuned model on the FULL training set, evaluate on the test set
best_rf = RandomForestRegressor(**study.best_params, n_jobs=-1, random_state=42)

with mlflow.start_run(run_name="random_forest_tuned"):
    best_rf.fit(X_train, y_train)
    preds = best_rf.predict(X_test)
    metrics = evaluate(y_test.values, preds)

    mlflow.log_param("model", "random_forest_tuned")
    mlflow.log_params(study.best_params)
    mlflow.log_metrics(metrics)

print("Tuned RandomForest on TEST set:")
for k, v in metrics.items():
    print(f"  {k:16}: {v:.6f}")

print("\nUntuned RF RMSE was: 0.004165")
print("Baseline RMSE was  : 0.004169")


🏃 View run random_forest_tuned at: https://dagshub.com/ar3080331/Crypto-Price-Forecasting-MLops.mlflow/#/experiments/0/runs/0d89a5c56f934e56a14ebed8adb2d2f7
🧪 View experiment at: https://dagshub.com/ar3080331/Crypto-Price-Forecasting-MLops.mlflow/#/experiments/0
Tuned RandomForest on TEST set:
  rmse            : 0.004161
  mae             : 0.002743
  r2              : 0.003676
  directional_acc : 0.505573

Untuned RF RMSE was: 0.004165
Baseline RMSE was  : 0.004169


## Step 5 — Register the tuned model to MLflow Registry (DagsHub)

We retrain the tuned RandomForest and log it **with the model artifact itself**,
then register it under a name so it becomes a versioned, deployable model.

- **Signature** — records the input columns/types so serving can validate inputs.
- **registered_model_name** — gives it an official name + auto-incrementing version
  (v1, v2, …) in the DagsHub Model Registry.


In [14]:
from mlflow.models import infer_signature

MODEL_NAME = "btc-rf-forecaster"

with mlflow.start_run(run_name="register-rf-tuned") as run:
    # retrain the tuned model (same best params)
    final_rf = RandomForestRegressor(**study.best_params, n_jobs=-1, random_state=42)
    final_rf.fit(X_train, y_train)

    preds = final_rf.predict(X_test)
    metrics = evaluate(y_test.values, preds)

    mlflow.log_param("model", "random_forest_tuned")
    mlflow.log_params(study.best_params)
    mlflow.log_metrics(metrics)

    # signature = the input/output schema, inferred from real data
    signature = infer_signature(X_train, final_rf.predict(X_train))

    mlflow.sklearn.log_model(
        sk_model=final_rf,
        name="model",
        signature=signature,
        input_example=X_train.head(3),
        registered_model_name=MODEL_NAME,
    )

print(f"Registered '{MODEL_NAME}'. Run ID: {run.info.run_id}")
print("Test metrics:", {k: round(v, 6) for k, v in metrics.items()})


Successfully registered model 'btc-rf-forecaster'.
2026/08/11 17:49:52 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: btc-rf-forecaster, version 1
Created version '1' of model 'btc-rf-forecaster'.


🏃 View run register-rf-tuned at: https://dagshub.com/ar3080331/Crypto-Price-Forecasting-MLops.mlflow/#/experiments/0/runs/26ecebd92b404cfb94a53d35243cdf63
🧪 View experiment at: https://dagshub.com/ar3080331/Crypto-Price-Forecasting-MLops.mlflow/#/experiments/0
Registered 'btc-rf-forecaster'. Run ID: 26ecebd92b404cfb94a53d35243cdf63
Test metrics: {'rmse': 0.004161, 'mae': 0.002743, 'r2': 0.003676, 'directional_acc': 0.505573}
